<a href="https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Warehouse connection ready")

Warehouse connection ready


In [14]:
df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,True


In [15]:
df["target"] = (df["gsc_clicks"] == 0).astype(int)

print(df["target"].value_counts())

target
1    3193080
0     417981
Name: count, dtype: int64


In [16]:
features = [
    "gsc_impressions",
    "gsc_avg_position"
]

X = df[features].copy()
y = df["target"].copy()

X = X.replace([float("inf"), float("-inf")], pd.NA)
X = X.fillna(X.median())

print("Features:", features)
print("X shape:", X.shape)

Features: ['gsc_impressions', 'gsc_avg_position']
X shape: (3611061, 2)


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

model_f1 = f1_score(
    y_test,
    model_pred,
    zero_division=0
)

print("Week-5 random split F1:", model_f1)

Week-5 random split F1: 0.9444241436735477


## 1. Two paper findings + my methodology questions


### Finding 1: The CTR Cliff

The paper reports that CTR changes strongly with search position, with a clear drop as pages move further down the search results.

**My methodology question:**  
Where exactly do the CTR, clicks, impressions, and position measurements come from, and are they measured over the same time period? I would also want to know whether the position buckets have enough observations to make the pattern stable.

I would treat this as an observed relationship between position and CTR rather than proof that position itself causes the CTR change.

### Finding 2: Content Refreshing

The paper reports that refreshing older content is associated with improved search performance.

**My methodology question:**  
How is a content refresh defined, and how is the outcome measured after the refresh? I would want to know whether the same pages are compared before and after the refresh and whether other changes during the same period could explain some of the observed improvement.

I would treat the finding as an observed pattern unless the methodology can separate the effect of refreshing from other factors.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)


In Week 5 I used a random train/test split. This can be optimistic because rows from the same client can appear in both the training and test sets.

For this audit, I will use a client-grouped split. All rows from one client will stay entirely in either the training set or the test set.

I will compare the Week-5 random-split result with the client-grouped result using the same Logistic Regression model, the same features, and the same F1 metric.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_f1 = f1_score(
    y_test,
    random_pred,
    zero_division=0
)

print("Week-5 random split F1:", random_f1)

Week-5 random split F1: 0.9444241436735477


In [20]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_hash_id"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

print("Training rows:", len(X_train_group))
print("Test rows:", len(X_test_group))

print(
    "Training clients:",
    df.iloc[train_idx]["client_hash_id"].nunique()
)

print(
    "Test clients:",
    df.iloc[test_idx]["client_hash_id"].nunique()
)

Training rows: 2690999
Test rows: 920062
Training clients: 37
Test clients: 10


In [21]:
group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(X_test_group)

group_f1 = f1_score(
    y_test_group,
    group_pred,
    zero_division=0
)

print("Week-6 client-grouped F1:", group_f1)

Week-6 client-grouped F1: 0.9431779395447242


In [22]:
validation_comparison = pd.DataFrame({
    "Validation": [
        "Week-5 random split",
        "Week-6 client-grouped split"
    ],
    "F1": [
        random_f1,
        group_f1
    ]
})

validation_comparison

,Validation,F1
0,Week-5 random split,0.944424
1,Week-6 client-grouped split,0.943178


## 3. Leakage audit

I checked the final feature set used by my Logistic Regression model.

The model uses two features: GSC impressions and GSC average position.

GSC clicks are not included as a feature because clicks were used to create the target. The target column itself is also not used as an input.

I did not use future-window information or product flags as model features.

One important limitation remains: the target is created from same-period GSC clicks. Therefore, this experiment should be treated as a modeling and validation exercise rather than a true future-performance prediction.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_audit = pd.DataFrame({
    "feature": features,
    "used_in_model": [True] * len(features),
    "target_derived": [
        feature in ["target", "gsc_clicks"]
        for feature in features
    ]
})

feature_audit

,feature,used_in_model,target_derived
0,gsc_impressions,True,False
1,gsc_avg_position,True,False


In [24]:
leaky_features = [
    feature
    for feature in features
    if feature in ["target", "gsc_clicks"]
]

print("Model features:", features)
print("Target column:", "target")
print("Target-derived features used:", leaky_features)
print("Future-window features used: none")
print("Product flags used: none")

Model features: ['gsc_impressions', 'gsc_avg_position']
Target column: target
Target-derived features used: []
Future-window features used: none
Product flags used: none


## 4. Claim rewrite

### Earlier claim

"The Logistic Regression model can identify content that needs attention."

### Safer claim

"On the March 2026 dataset, Logistic Regression achieved a measured F1 score of 0.9444 under the original random split. Under a stricter client-grouped split, the measured F1 was 0.9432.

The small difference between the two validation designs suggests that the model's measured performance was relatively stable for this dataset. However, this is directional evidence and should be treated as decision-support rather than proof of production performance.

I also found that my Week-5 baseline comparison was circular because the target was created directly from GSC clicks and the baseline used the same signal. Therefore, I do not claim that the model beat the Week-4 baseline.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

group_errors = pd.DataFrame({
    "actual": y_test_group,
    "predicted": group_pred
})

group_errors["correct"] = (
    group_errors["actual"] == group_errors["predicted"]
)

print(
    "Incorrect predictions:",
    (~group_errors["correct"]).sum()
)

group_errors[
    ~group_errors["correct"]
].head(10)

Incorrect predictions: 95558


,actual,predicted,correct
2,0,1,False
5,0,1,False
17,0,1,False
59,0,1,False
115,0,1,False
134,0,1,False
301,0,1,False
302,0,1,False
303,1,0,False
305,0,1,False


### Error review

I inspected incorrect predictions from the client-grouped test set.

The errors show that impressions and average position do not perfectly separate the two target classes. Some rows with similar search signals can still have different target labels.

These errors are useful for understanding where the model is uncertain. They should be treated as cases for further investigation rather than automatic content decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.